# MDA Text Preprocessing Pipeline

**Input:** Folder of MDA `.txt` files named like `Alphabet Inc._10-Q_2025-09-30T00_00_00_English_MDA.txt`

**Output:** Two DataFrames:
1. **Document-level** (`df_doc`) — one row per filing, for company-level sentiment & topic modelling
2. **Sentence-level** (`df_sent`) — one row per sentence, for joint sentiment × topic analysis

## Imports & Config

In [9]:
import re
import os
import pandas as pd
import nltk
from pathlib import Path
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# ── Config ──────────────────────────────────────────────────────────
MDA_FOLDER       = "../MDA"          
FILE_PATTERN     = "*_MDA.txt"        
MIN_SENT_TOKENS  = 5                
BIGRAM_MIN_COUNT = 10                 # n-gram: pair must appear ≥ this many times
BIGRAM_THRESHOLD = 3.0                # n-gram: NPMI-like score threshold


## Stop Words & Regex Patterns
All patterns carried over from the NVIDIA notebook, with company-specific terms generalised.

In [ ]:
# ── Stop words ──────────────────────────────────────────────────────
FINANCE_STOPWORDS = {
    "herein", "thereof", "thereto", "therein", "hereby", "pursuant",
    "accordance", "aforementioned", "foregoing", "whereas", "whereby",
    "hereafter", "hereinafter",
    "form", "quarterly", "annual", "report", "filing", "fiscal",
    "quarter", "period", "ended", "condensed", "consolidated",
    "statements", "notes", "refer", "also", "following", "described",
    "set", "forth", "part", "item",
    "incorporated", "reincorporated", "headquartered", "mean", "means",
    "including", "included", "related", "thereto", "million", "thousand", "billion",
    "company", "corporation", "subsidiaries", "inc", "ltd", "limited",
    "platforms", "manufacturing"
}

GEO_STOPWORDS = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
    "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
    "maine", "maryland", "massachusetts", "michigan", "minnesota",
    "mississippi", "missouri", "montana", "nebraska", "nevada",
    "new_hampshire", "new_jersey", "new_mexico", "new_york", "north_carolina",
    "north_dakota", "ohio", "oklahoma", "oregon", "pennsylvania",
    "rhode_island", "south_carolina", "south_dakota", "tennessee", "texas",
    "utah", "vermont", "virginia", "washington", "west_virginia",
    "wisconsin", "wyoming",
    "ca", "de", "tx", "ny", "wa", "nv", "fl", "ga", "il", "ma",
    "san_francisco", "san_jose", "santa_clara", "santa", "clara",
    "palo_alto", "menlo_park", "mountain_view", "sunnyvale", "cupertino",
    "redwood_city", "fremont", "san_diego", "los_angeles", "seattle",
    "portland", "austin", "boston", "new_york_city", "new_york",
    "chicago", "denver", "atlanta", "miami",
    "delaware", "nevada", "cayman_islands", "bermuda", "ireland",
    "luxembourg", "singapore", "hong_kong", "switzerland",
    "united_states", "united_kingdom", "china", "japan", "germany",
    "france", "india", "taiwan", "south_korea", "israel", "canada",
    "australia", "netherlands", "sweden", "finland",
    "north_america", "south_america", "latin_america", "europe",
    "asia_pacific", "middle_east", "apac", "emea", "americas",
}


BOILERPLATE_SENTENCE_PATTERNS = [
    # --- Cross-references & Navigation ("please see note X") ---
    re.compile(r'(?:please[,\s]*)?note[,\s]*financial[,\s]*(?:additional[,\s]*)?(?:information|discussion|disclosure)'),
    re.compile(r'please[,\s]*(?:see[,\s]*)?note[,\s]*financial'),
    re.compile(r'adoption[,\s]*new[,\s]*recently[,\s]*issued[,\s]*accounting[,\s]*pronouncements'),
    re.compile(r'recently[,\s]*issued[,\s]*accounting[,\s]*pronouncements'),
    re.compile(r'additional[,\s]*factors[,\s]*see'),
    re.compile(r'contents[,\s]*website'),
    re.compile(r'internet[,\s]*address[,\s]*www'),
    re.compile(r'headquarter[,\s]*facilities'),
    # --- Accounting Policy Boilerplate ---
    re.compile(r'critical[,\s]*accounting[,\s]*policies[,\s]*estimates'),
    re.compile(r'managements[,\s]*discussion[,\s]*analysis[,\s]*financial[,\s]*condition[,\s]*results[,\s]*operations'),
    re.compile(r'preparation_financial[,\s]*requires[,\s]*us[,\s]*make[,\s]*estimates[,\s]*judgments'),
    re.compile(r'base[,\s]*estimates[,\s]*historical_experience[,\s]*various_assumptions[,\s]*believed[,\s]*reasonable'),
    re.compile(r'believe[,\s]*critical_accounting[,\s]*policies[,\s]*affect[,\s]*significant[,\s]*judgments'),
    re.compile(r'management[,\s]*discussed[,\s]*development[,\s]*selection[,\s]*critical[,\s]*accounting'),
    re.compile(r'audit_committee[,\s]*reviewed[,\s]*disclosures[,\s]*relating[,\s]*critical[,\s]*accounting'),
    re.compile(r'on-going[,\s]*basis[,\s]*evaluate[,\s]*estimates'),
    # --- Revenue Recognition Boilerplate ---
    re.compile(r'recognize[,\s]*revenue[,\s]*product[,\s]*sales[,\s]*persuasive[,\s]*evidence'),
    re.compile(r'consider[,\s]*delivery[,\s]*occur[,\s]*upon[,\s]*shipment[,\s]*provided[,\s]*title[,\s]*risk[,\s]*loss'),
    re.compile(r'point[,\s]*sale[,\s]*assess[,\s]*whether[,\s]*arrangement[,\s]*fee[,\s]*fixed[,\s]*determinable'),
    re.compile(r'determine[,\s]*collection[,\s]*fee[,\s]*reasonably_assured[,\s]*defer'),
    re.compile(r'policy[,\s]*sales[,\s]*certain[,\s]*distributors[,\s]*rights[,\s]*return[,\s]*defer'),
    re.compile(r'unclaimed[,\s]*rebates[,\s]*reversed[,\s]*revenue'),
    re.compile(r'rebates[,\s]*typically[,\s]*expire[,\s]*six[,\s]*months'),
    re.compile(r'accrue[,\s]*potential[,\s]*rebates[,\s]*apply[,\s]*breakage[,\s]*factor'),
    # --- Tax & Goodwill Impairment Boilerplate ---
    re.compile(r'income_taxes[,\s]*recognize[,\s]*federal[,\s]*state[,\s]*foreign[,\s]*current[,\s]*tax'),
    re.compile(r'recognize[,\s]*federal[,\s]*state[,\s]*foreign[,\s]*deferred[,\s]*tax[,\s]*assets[,\s]*liabilities'),
    re.compile(r'united_states[,\s]*income_tax[,\s]*provided[,\s]*earnings[,\s]*non-us'),
    re.compile(r'recognize[,\s]*benefit[,\s]*tax_position[,\s]*more-likely-than-not'),
    re.compile(r'policy[,\s]*include[,\s]*interest_penalties[,\s]*unrecognized_tax_benefits'),
    re.compile(r'goodwill_impairment[,\s]*review[,\s]*process[,\s]*compares[,\s]*fair[,\s]*value'),
    re.compile(r'goodwill[,\s]*subject[,\s]*impairment[,\s]*test[,\s]*fourth'),
    re.compile(r'utilize[,\s]*two-step[,\s]*approach[,\s]*testing[,\s]*goodwill_impairment'),
    re.compile(r'performed[,\s]*impairme'), 
    # --- Investment & Securities Boilerplate ---
    re.compile(r'cash[,\s]*equivalents[,\s]*marketable[,\s]*securities[,\s]*(?:cash_equivalents[,\s]*)?consist'),
    re.compile(r'generally[,\s]*classify[,\s]*marketable_securities[,\s]*date[,\s]*acquisition'),
    re.compile(r'securities_reported_fair[,\s]*value_unrealized_gains[,\s]*losses[,\s]*accumulated'),
    re.compile(r'investment[,\s]*policy[,\s]*requires[,\s]*purchase[,\s]*(?:top-tier|high_grade|highly_rated)'),
    re.compile(r'investments[,\s]*(?:aa2|aa3)[,\s]*better_rated_securities'),
    re.compile(r'investments[,\s]*investments[,\s]*auction-rate[,\s]*preferred'),
    re.compile(r'cash[,\s]*equivalents[,\s]*marketable[,\s]*securities[,\s]*treated[,\s]*available-for-sale'),
    re.compile(r'available-for-sale[,\s]*investments[,\s]*subject[,\s]*periodic[,\s]*impairment_review'),
    re.compile(r'fixed_rate[,\s]*debt_securities[,\s]*may[,\s]*market_value[,\s]*adversely_impacted'),
    re.compile(r'may[,\s]*suffer[,\s]*losses[,\s]*principal[,\s]*forced[,\s]*sell[,\s]*securities'),
    re.compile(r'realized_gains[,\s]*losses_sale[,\s]*marketable_securities[,\s]*determined[,\s]*using[,\s]*specific'),

    # --- Stock Repurchase & Equity Boilerplate ---
    re.compile(r'repurchases_made[,\s]*open[,\s]*market[,\s]*privately_negotiated_transactions'),
    re.compile(r'program[,\s]*obligate[,\s]*acquire[,\s]*particular[,\s]*amount[,\s]*common_stock'),
    re.compile(r'share_repurchase_program[,\s]*entered_may_continue[,\s]*enter'),
    re.compile(r'agreements[,\s]*generally[,\s]*require[,\s]*make[,\s]*up-front[,\s]*payment[,\s]*exchange'),

    # --- Accounts Receivable & Risk Boilerplate ---
    re.compile(r'accounts[,\s]*receivable[,\s]*highly[,\s]*concentrated[,\s]*make[,\s]*us_vulnerable'),
    re.compile(r'maintain[,\s]*allowance[,\s]*doubtful[,\s]*accounts[,\s]*(?:receivable[,\s]*)?estimated[,\s]*losses'),
    re.compile(r'allowance[,\s]*consists[,\s]*amount[,\s]*identified[,\s]*specific_customers'),
    re.compile(r'financial_condition_customers[,\s]*(?:financial_institutions[,\s]*)?(?:providing[,\s]*)?.*deteriorate[,\s]*resulting[,\s]*impairment'),
    re.compile(r'strive[,\s]*limit[,\s]*exposure[,\s]*uncollectible[,\s]*accounts_receivable'),
    re.compile(r'difficulties[,\s]*heightened[,\s]*periods[,\s]*economic_conditions[,\s]*worsen'),
    re.compile(r'continue_work[,\s]*directly[,\s]*foreign[,\s]*customers[,\s]*may[,\s]*difficult'),

    # --- Inventory Valuation Boilerplate ---
    re.compile(r'inventories[,\s]*inventory[,\s]*cost[,\s]*computed[,\s]*adjusted[,\s]*standard[,\s]*basis'),
    re.compile(r'write[,\s]*inventory[,\s]*(?:estimated[,\s]*)?lower[,\s]*cost[,\s]*(?:estimated[,\s]*)?market'),
    re.compile(r'inventory_reserves[,\s]*established[,\s]*reversed[,\s]*inventory[,\s]*sold[,\s]*scrapped'),

    # --- Stock-Based Compensation Boilerplate ---
    re.compile(r'stock-based_compensation[,\s]*(?:stock-based_compensation[,\s]*)?cost[,\s]*(?:equity[,\s]*awards[,\s]*)?measured[,\s]*grant[,\s]*date'),
    re.compile(r'recognize[,\s]*stock-based_compensation_expense[,\s]*using[,\s]*straight-line'),
    re.compile(r'estimate_fair_value[,\s]*employee_stock[,\s]*options[,\s]*date[,\s]*grant[,\s]*using[,\s]*binomial'),
    re.compile(r'forfeitures[,\s]*estimated[,\s]*based[,\s]*historical_experience'),
    re.compile(r'risk-free[,\s]*interest_rate[,\s]*assumption[,\s]*based_upon[,\s]*observed'),
    re.compile(r'dividend_yield[,\s]*assumption[,\s]*based[,\s]*history[,\s]*expectation'),

    # --- Litigation Boilerplate ---
    re.compile(r'aggressively[,\s]*defending[,\s]*current[,\s]*litigation[,\s]*matters'),
    re.compile(r'however[,\s]*many[,\s]*uncertainties[,\s]*associated[,\s]*litigation_investigations[,\s]*cannot'),
    re.compile(r'information[,\s]*becomes[,\s]*available[,\s]*causes[,\s]*us[,\s]*determine[,\s]*loss[,\s]*pending'),

    # --- Liquidity & Capital Resources Boilerplate ---
    re.compile(r'liquidity[,\s]*primary[,\s]*source[,\s]*liquidity[,\s]*cash_generated'),
    re.compile(r'however[,\s]*assurance[,\s]*need[,\s]*raise[,\s]*additional_equity_debt'),
    re.compile(r'additional_financing[,\s]*may_available_favorable[,\s]*terms[,\s]*may[,\s]*dilutive'),
    re.compile(r'may_require_additional[,\s]*capital[,\s]*purposes[,\s]*presently[,\s]*contemplated'),
    re.compile(r'unable_obtain_sufficient[,\s]*capital[,\s]*could_required_curtail'),
    re.compile(r'believe_existing[,\s]*cash_balances[,\s]*anticipated[,\s]*cash[,\s]*flows'),

    # --- Section Headers & Structural Markers ---
    re.compile(r'^(?:table_contents|table[,\s]*co(?:ntents)?|table[,\s]*conten)'),
    re.compile(r'^(?:results_operations|results[,\s]*operations)$'),
    re.compile(r'^(?:information)$'),
    re.compile(r'^(?:operating_segments[,\s]*equivalent)$'),
    re.compile(r'risk_factors_risks[,\s]*business[,\s]*industry[,\s]*partners'),
    re.compile(r'recent[,\s]*developments[,\s]*future[,\s]*objectives[,\s]*challenges'),
    re.compile(r'material[,\s]*off-balance[,\s]*sheet[,\s]*arrangements[,\s]*defined[,\s]*regulation'),
    re.compile(r'contractual[,\s]*obligations[,\s]*material[,\s]*changes[,\s]*outside[,\s]*ordinary'),

    # --- Product Defect / Warranty Boilerplate ---
    re.compile(r'products[,\s]*complex[,\s]*may[,\s]*contain[,\s]*defects[,\s]*experience[,\s]*failures'),
    re.compile(r'products[,\s]*technologies[,\s]*contains[,\s]*defect[,\s]*compatibility[,\s]*issue'),
    re.compile(r'efforts[,\s]*could[,\s]*divert[,\s]*managements[,\s]*engineers[,\s]*attention'),
    re.compile(r'product[,\s]*recall[,\s]*significant[,\s]*number[,\s]*product_returns[,\s]*could[,\s]*expensive'),
    re.compile(r'may_required[,\s]*reimburse[,\s]*customers[,\s]*.*costs[,\s]*repair[,\s]*replace'),
    re.compile(r'warranty[,\s]*liabilities[,\s]*cost_revenue_includes[,\s]*estimated[,\s]*cost[,\s]*product[,\s]*warranties'),

    # --- Geographic & Segment Disclosure Boilerplate ---
    re.compile(r'revenue[,\s]*geographic[,\s]*region[,\s]*allocated[,\s]*individual[,\s]*countries'),
    re.compile(r'sales_customers_outside[,\s]*united[,\s]*states'),
    re.compile(r'revenue_attributable_end[,\s]*customers_different_location'),
    re.compile(r'revenue_significant_customers[,\s]*representing_total_revenue'),
    re.compile(r'financial_information_business[,\s]*segment_geographic_data'),
    re.compile(r'gross_margin_percentage[,\s]*gross_profit[,\s]*revenue'),
    re.compile(r'gross[,\s]*profit[,\s]*consists[,\s]*total[,\s]*revenue_net_allowances[,\s]*less_cost_revenue'),

    # --- Cost of Revenue Definition Boilerplate ---
    re.compile(r'cost[,\s]*revenue[,\s]*consists[,\s]*primarily[,\s]*cost[,\s]*semiconductors[,\s]*purchased[,\s]*subcontractors'),
    re.compile(r'cost_revenue_includes[,\s]*development_costs[,\s]*license[,\s]*service[,\s]*arrangements'),

]

MDA_BOILERPLATE_STOPWORDS = {
    # Accounting & Reporting
    "accounting", "adoption", "analysis", "applicable", "arrangements",
    "assessment", "audit_committee", "based", "believe", "certain",
    "changes", "compliance", "condition", "consists", "contingencies",
    "critical_accounting", "deferred", "disclosure", "discussion", "effective",
    "effective_tax_rate", "election", "estimate", "estimates", "estimates judgments",
    "expense", "expenses", "factors", "financial", "follows",
    "general", "gross", "however", "impairme", "include",
    "includes", "income", "income tax", "income taxes", "information",
    "issued", "judgments", "less", "liability", "managements",
    "material", "net", "net income", "note", "obligations",
    "operating", "operating_expenses", "operations", "percentage", "periods",
    "plan", "please", "policies", "policy", "preparation_financial",
    "primarily", "primarily_due", "primarily_driven", "prior", "provided",
    "provisions", "rates", "recognized", "recorded", "recently",
    "reflecting", "regarding", "reported", "required", "requires",
    "resources", "respectively", "result", "results", "results_operations",
    "revenue", "sales", "segment", "service", "significant",
    "states", "subject", "sufficient", "tax", "taxes",
    "total", "types", "unable", "united", "united_states",
    "us", "used", "using", "value", "well",
    "within", "would",

    # Financial Instruments & Investments
    "agencies", "available", "balances", "billion", "billion_cash",
    "cash", "cash_cash_equivalents", "cash_dividends", "commercial_paper", "coupon",
    "credit", "currency", "debt", "discount_amortization", "dollars",
    "end_year", "equivalents", "existing", "federal", "financing",
    "financing_activities", "fixed_rate", "foreign", "held", "highly",
    "interest", "interest_expense", "interest_income", "investing", "investing_activities",
    "investment", "investments", "liquidity", "marketable", "marketable_securities",
    "million", "net_cash_provided", "net_cash_used", "outstanding", "pay",
    "payment", "payments", "portfolio", "purchase", "purchases",
    "securities",

    # Corporate Actions & Equity
    "authorized", "board", "capital", "capital_return_shareholders", "common",
    "dividend", "future_cash_dividends", "program", "program_subject", "repurchase",
    "repurchased", "share", "shares", "stock", "stock-based",
    "stock-based_compensation",

    # Generic Business Language
    "additional", "agreements", "amount", "anticipated", "associated",
    "benefits", "combination", "compared", "continued", "cost",
    "costs", "could", "customer", "customers", "data",
    "date", "decrease", "decreased", "demand", "development",
    "driven", "due", "effect", "end", "enterprise",
    "expected", "expect", "first", "first_year", "four",
    "future", "growth", "higher", "impact", "increase",
    "increased", "increases", "individual", "large", "lower",
    "made", "manufacturing", "market", "markets", "may",
    "mix", "months", "next", "new", "offset",
    "overall", "overall_gross_margin", "partners", "partially_offset", "performance",
    "periods", "platforms", "previously", "primary", "product",
    "products", "production", "region", "requirements", "second",
    "second_year", "services", "solutions", "sources", "support",
    "systems", "technology", "testing", "time", "two",
    "use", "without", "year", "years",

    # Temporal Boilerplate Markers
    "first_year", "second_year", "half", "nine", "months",
    "first", "second", "third", "fourth", "year_compared",
    "year_respectively", "year_primarily_due",

    # Section Headers (remove entire tokens)
    "table_contents", "table", "conten", "rese", "genera",
    "fisc", "followi", "percenta", "componen",

    # NVIDIA-Specific Boilerplate (review if analyzing other companies)
    "gpu", "gpus", "geforce", "quadro", "tegra",
    "automotive", "gaming", "graphics", "processor", "processors",
    "computing", "architecture", "platform", "platforms", "mobile",
    "cloud", "comprised", "challenges",

    # Geographic Boilerplate
    "accounted", "allocated", "billed", "concentration_revenue_revenue", "countries",
    "customers_different_location", "even", "geographic", "initially", "location",
    "revenue_attributable_end", "sales_customers_outside",

    # Discussion Section Headers
    "discussion_gross_margin", "discussion", "overview", "recent", "developments",
    "objectives"
    }

STOPWORDS = set(stopwords.words("english")) | FINANCE_STOPWORDS | GEO_STOPWORDS

In [ ]:
# ── FLS boundary ───────────────────────────────────────────────────
FLS_END_ANCHORS = [
    # Company overview variants
    r"Overview\s+Our\s+Company\s+and\s+Our\s+Businesses",
    r"Overview\s+Our\s+Company",
    r"Business\s+Overview\s",
    r"Company\s+Overview\s",
    r"Corporate\s+Overview\s",
    r"General\s+Overview\s",
    r"Overview\s+of\s+the\s+Company\s",
    r"Overview\s+of\s+Our\s+Business\s",
    r"Overview\s",

    # Executive summary
    r"Executive\s+Summary\s",
    r"Executive\s+Overview\s",
    r"Management\s+Summary\s",

    # Business / general
    r"Business\s+General\s",
    r"General\s",
    r"Introduction\s",
    r"Background\s",
    r"Results\s+of\s+Operations",

    # Industry / market context
    r"Industry\s+Overview\s",
    r"Market\s+Overview\s",
    r"Our\s+Business\s",
    r"Our\s+Company\s",
    r"About\s+the\s+Company\s",
    r"Who\s+We\s+Are\s",
]

_FLS_PATTERN = re.compile("|".join(FLS_END_ANCHORS), re.IGNORECASE)
print(_FLS_PATTERN)

# ── Table removal: atomic building blocks ──────────────────────────
_VAL = (
    r'(?:\$\s*)?-?[\d,]+(?:\.\d+)?\s*(?:%|pts)?'
    r'|\([0-9,\.]+\)\s*(?:%|pts)?'
)
_DATE = (
    r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    r'\s+\d{1,2},?\s*\d{4}'
)
_PERIOD = (
    r'Three\s+Months\s+Ended'
    r'|Six\s+Months\s+Ended'
    r'|Nine\s+Months\s+Ended'
    r'|Twelve\s+Months\s+Ended'
    r'|Quarter-over-Quarter\s+Change'
    r'|Year-over-Year\s+Change'
    r'|\$\s*Change'
    r'|%\s*Change'
)
_UNITS = r'\(\$?\s*[Ii]n\s+(?:millions|billions)[^)]*\)'

# ── Table removal: compiled patterns ───────────────────────────────
RE_HEADER_ROW = re.compile(
    r'(?:{date}|{period}|{units})'
    r'(?:\s+(?:{date}|{period}|{units}))*'
    .format(date=_DATE, period=_PERIOD, units=_UNITS),
    re.IGNORECASE
)
RE_DATA_ROW = re.compile(
    r'(?<![.!?]\s)'
    r'[A-Za-z][A-Za-z0-9 \&\(\),\-\.]{{1,50}}'
    r'(?:\s+(?:{val})){{2,}}'
    .format(val=_VAL),
    re.IGNORECASE
)
RE_FOLLOWING_TABLE = re.compile(r'[Tt]he following table[^.]+\.', re.IGNORECASE)
RE_ORPHAN_NUMS = re.compile(
    r'(?<!\w)(?:{val})(?:\s+(?:{val})){{0,3}}(?!\s*\w{{4}})'.format(val=_VAL),
    re.IGNORECASE
)
RE_PAGE_NUM = re.compile(r'\b\d{1,3}\b(?=\s+[A-Z])')

# ── Structural noise ───────────────────────────────────────────────
RE_ITEM_TAG   = re.compile(r'\bItem\s+\d+[A-Z]?\.\s*')
RE_PART_TAG   = re.compile(r'\bPart\s+[IVX]+,?\s*')
RE_COPYRIGHT  = re.compile(r'©?\s*\d{4}\s+\w[^.]*(?:Corporation|Inc|LLC|Ltd)[^.]*\.')
RE_WHITESPACE = re.compile(r'\s{2,}')

# ── N-gram blacklist ───────────────────────────────────────────────
NGRAM_BLACKLIST_PATTERNS = [
    r'^\s*[,;:\-\(\)]+',
    r'[,;:\-\(\)]+\s*$',
    r'^-\w+',
    r'\b(\w{4,})_\1\b',
    r'\b(\w{4,})_\w+_\1\b',
    r'(\b\w+_\w+)_\w*_?\1',
    r'\b(santa_clara|san_jose|palo_alto|menlo_park|santa_monica)\b',
    r'\b(california|delaware|nevada|washington|new_york|texas|georgia)\b',
    r'\b(san|santa|los|las|new|mount|fort|north|south|east|west)_\b',
    r'\bsan_(francisco|jose|diego|antonio|mateo|bernardino)\b',
    r'\bsanta_(clara|monica|barbara|ana|cruz|rosa)\b',
    r'\blos_(angeles|altos|gatos)\b',
    r'\bnew_(york|jersey|mexico|hampshire|orleans|haven)\b',
    r'\bfort_(worth|lauderdale|wayne|collins)\b',
    r'\bmount_(view|pleasant)\b',
    r'\b(california|delaware|nevada|washington|texas).*\b(california|delaware|nevada|washington|texas)\b',
    r'\bnote_financial\b',
    r'\b(form_10|10_k|10_q|8_k)\b',
    r'\b(exhibit|item_\d+|part_[ivx]+)\b',
    r'\b(one|two|three|four|five|reportable)_segment',
    r'\breporting_segment',
    r'\b(entered|continue|remained|could|would|should)_may\b',
    r'\bmay_(continue|remain|result|impact|affect|cause)\b',
    r'\b(first|second|third|fourth)_quarter_\w+_quarter\b',
    r'^\d+_',
]

re.compile('Overview\\s+Our\\s+Company\\s+and\\s+Our\\s+Businesses|Overview\\s+Our\\s+Company|Business\\s+Overview\\s|Company\\s+Overview\\s|Corporate\\s+Overview\\s|General\\s+Overview\\s|Overview\\s+of\\s+the\, re.IGNORECASE)


## Helper Functions

In [12]:
def parse_filename(filename: str) -> dict:
    """
    Parse: Alphabet Inc._10-Q_2025-09-30T00_00_00_English_MDA.txt
    Company name = everything before the first _10-K or _10-Q.
    Returns dict with: company_name, filing_type, filing_date, year, quarter
    """
    stem = Path(filename).stem
    m = re.match(
        r'^(?P<company>.+)_(?P<ftype>10-[KQ])_(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})(?:T[\d_]+)?_(?P<section>.+)',
        stem
    )
    if not m:
        return None

    company_name = m.group('company')
    filing_type  = m.group('ftype')
    year         = m.group('year')
    month        = int(m.group('month'))
    day          = m.group('day')
    quarter      = f"Q{(month - 1) // 3 + 1}"
    filing_date  = f"{year}-{m.group('month')}-{day}"

    return {
        "company_name": company_name,
        "filing_type":  filing_type,
        "filing_date":  filing_date,
        "year":         year,
        "quarter":      quarter,
    }

def is_blacklisted(ngram: str) -> bool:
    return any(re.search(pat, ngram, re.IGNORECASE) for pat in NGRAM_BLACKLIST_PATTERNS)

def strip_fls(raw: str, filename: str = "") -> str:
    match = _FLS_PATTERN.search(raw)
    if match:
        return raw[match.start():]
    else: 
        print (f"FLS anchor not found in '{filename}' - using full doc.")
        return raw 


def remove_tables(text: str) -> str:
    text = RE_HEADER_ROW.sub(' ', text)
    text = RE_DATA_ROW.sub(' ', text)
    text = RE_FOLLOWING_TABLE.sub(' ', text)
    text = RE_ORPHAN_NUMS.sub(' ', text)
    text = RE_PAGE_NUM.sub(' ', text)
    return text


def clean(text: str) -> str:
    text = remove_tables(text)
    text = RE_ITEM_TAG.sub(' ', text)
    text = RE_PART_TAG.sub(' ', text)
    text = RE_COPYRIGHT.sub(' ', text)
    text = RE_WHITESPACE.sub(' ', text)
    return text.strip()


def get_sentences(text: str) -> list:
    return [s.strip() for s in sent_tokenize(text)
            if len(s.split()) >= MIN_SENT_TOKENS]


def normalise(sentence: str, remove_stops: bool = False) -> list:
    sentence = sentence.lower()
    sentence = re.sub(r"[^\w\s\-]", "", sentence)
    tokens   = sentence.split()
    tokens   = [t for t in tokens if not re.fullmatch(r'[\d,\.\-\%\$]+', t)]
    if remove_stops:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

## N-gram Detection (gensim Phrases)
### gensim better than for loops for 20k doc, anf i think spacy is overkill for js text cleaning (and i dont hv gpu)


In [13]:
def build_phrase_models(all_token_lists):
    """
    Train bigram and trigram models on the full corpus.
    connector_words lets gensim bridge over 'of', 'the', 'and' etc.
    so 'results of operations' → 'results_of_operations'.
    """
    bigram = Phrases(
        all_token_lists,
        min_count=BIGRAM_MIN_COUNT,
        threshold=BIGRAM_THRESHOLD,
        connector_words=ENGLISH_CONNECTOR_WORDS,
    )
    trigram = Phrases(
        bigram[all_token_lists],
        min_count=BIGRAM_MIN_COUNT,
        threshold=BIGRAM_THRESHOLD,
        connector_words=ENGLISH_CONNECTOR_WORDS,
    )
    return bigram, trigram


def apply_phrases(tokens, bigram, trigram) -> list:
    """
    Apply bigram → trigram models, then blacklist filter.
    Hard cap: split any n-gram > 3 tokens back to unigrams.
    """
    phrased = list(trigram[bigram[tokens]])
    result = []
    for tok in phrased:
        parts = tok.split('_')
        if len(parts) <= 3 and not is_blacklisted(tok):
            result.append(tok)
        elif len(parts) > 3:
            result.extend(p for p in parts if not is_blacklisted(p))
        # else: blacklisted, drop it
    return result

## Main Pipeline
Reads all MDA files, cleans them, builds n-gram models on the full corpus, and produces two DataFrames.

In [14]:
def run_pipeline(mda_folder: str, file_pattern: str):
    """
    Returns:
        df_doc  — one row per filing (for company-level sentiment & topic modelling)
        df_sent — one row per sentence (for sentence-level sentiment × topic)
    """
    mda_path = Path(mda_folder)
    files = sorted(mda_path.glob(file_pattern))
    print(f"Found {len(files)} MDA files in {mda_folder}")

    if not files:
        raise FileNotFoundError(f"No files matching '{file_pattern}' in {mda_folder}")

    # ── Pass 1: read, parse metadata, clean, tokenise ──────────────
    print("Pass 1: reading and tokenising...")
    doc_store = {}  # keyed by filepath

    for fp in files:
        meta = parse_filename(fp.name)
        if meta is None:
            print(f"  SKIPPED (bad filename): {fp.name}")
            continue

        raw_text = fp.read_text(encoding='utf-8', errors='replace')
        text     = clean(strip_fls(raw_text, fp))
        sents    = get_sentences(text)

        if not sents:
            print(f"  SKIPPED (no sentences after cleaning): {fp.name}")
            continue

        doc_id = f"{meta['company_name']}_{meta['filing_type']}_{meta['filing_date']}"

        doc_store[doc_id] = {
            "meta":       meta,
            "sentences":  sents,
            "raw_text":   text,
            "sent_tokens_clean": [normalise(s, remove_stops=True)  for s in sents],
            "sent_tokens_raw":   [normalise(s, remove_stops=False) for s in sents],
        }

    print(f"  {len(doc_store)} documents loaded successfully")

    # ── Train n-gram models on the FULL corpus ─────────────────────
    print("Training bigram/trigram models across full corpus...")
    all_clean_tokens = [tl for doc in doc_store.values()
                           for tl in doc["sent_tokens_clean"]]

    bigram, trigram = build_phrase_models(all_clean_tokens)
    print(f"  Phrase models trained on {len(all_clean_tokens)} sentences")

    # ── Pass 2: build DataFrame ──────────────────────────────
    print("Pass 2: building output DataFrames...")
    doc_rows  = []
    for doc_id, doc in doc_store.items():
        meta = doc["meta"]

        # Apply phrases to each sentence's clean tokens
        phrased_per_sent = [
            apply_phrases(tl, bigram, trigram)
            for tl in doc["sent_tokens_clean"]
        ]

        # ── Document-level row ─────────────────────────────────────
        all_clean_tokens_flat = [tok for sent_toks in phrased_per_sent
                                     for tok in sent_toks]

        doc_rows.append({
            "doc_id":       doc_id,
            "company_name": meta["company_name"],
            "filing_type":  meta["filing_type"],
            "filing_date":  meta["filing_date"],
            "year":         meta["year"],
            "quarter":      meta["quarter"],
            "raw_text":     doc["raw_text"],
            "sentences":    doc["sentences"],        # list[str]
            "clean_tokens": all_clean_tokens_flat,    # flat list[str]
            "n_sentences":  len(doc["sentences"]),
            "n_tokens":     len(all_clean_tokens_flat),
        })

    df_doc  = pd.DataFrame(doc_rows)
    # Sort for consistent output
    df_doc  = df_doc.sort_values(["company_name", "filing_date"]).reset_index(drop=True)

    print(f"\nDone!")
    print(f"df_doc:  {df_doc.shape[0]} documents × {df_doc.shape[1]} columns")
    return df_doc

## Run & Export

In [15]:
df_doc= run_pipeline(MDA_FOLDER, FILE_PATTERN)

# ── Quick inspection ───────────────────────────────────────────────
print("\n=== df_doc (first 3 rows, key columns) ===")
print(df_doc[["doc_id", "company_name", "filing_type", "filing_date",
              "n_sentences", "n_tokens"]].head(3).to_string(index=False))

# ── Coverage summary ───────────────────────────────────────────────
print("\n=== Documents per company ===")
print(df_doc.groupby("company_name")["doc_id"].count().to_string())

Found 18183 MDA files in ../MDA
Pass 1: reading and tokenising...
  18026 documents loaded successfully
Training bigram/trigram models across full corpus...
  Phrase models trained on 5006435 sentences
Pass 2: building output DataFrames...

Done!
df_doc:  18026 documents × 11 columns

=== df_doc (first 3 rows, key columns) ===
                       doc_id  company_name filing_type filing_date  n_sentences  n_tokens
1-800-PetMeds_10-Q_2020-01-28 1-800-PetMeds        10-Q  2020-01-28          115      1046
1-800-PetMeds_10-K_2020-05-26 1-800-PetMeds        10-K  2020-05-26          150      1428
1-800-PetMeds_10-Q_2020-08-03 1-800-PetMeds        10-Q  2020-08-03          103       896

=== Documents per company ===
company_name
1-800-PetMeds                              17
3D_Systems                                 44
8x8                                        44
A10_Networks                               44
ACI_Worldwide                              63
ACM_Research                     

In [16]:
# ── Save to csv (preserves list columns natively) ──────────────
df_doc.to_csv("mda_doc_level.csv",  index=False)
print("Saved: mda_doc_level.csv")

df_doc.head(50).to_csv("mda_doc_level_top50.csv",  index=False)


Saved: mda_doc_level.csv
